# Rule of Thumb — tabular quickstart

This notebook demonstrates the tabular RoT explainer on a small synthetic
problem: we define a toy "black box" classifier, fit a Rule-of-Thumb
surrogate to its outputs, and inspect the learned feature importances.

Everything runs on CPU with synthetic data in a few seconds.

In [1]:
import numpy as np

from ruleofthumb import RuleOfThumb

In [2]:
# Synthetic data and a trivially simple "black box"
rng = np.random.RandomState(0)
n, d = 2000, 5
X = rng.randn(n, d).astype(np.float32)

def black_box(X):
    # logistic rule driven by features 0 and 2 only
    z = 2.0 * X[:, 0] - 1.5 * X[:, 2]
    return (1 / (1 + np.exp(-z)) > 0.5).astype(np.int64)  # int labels

y = black_box(X)

In [3]:
rot = RuleOfThumb(y_outputs=y, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05)
importances = rot.get_explanation(X)  # signed: positive = evidence toward class 1
importances[:3]

array([[ 2.4874446 ,  0.01250004, -0.88556355, -0.08100072, -0.07325435],
       [-1.3206879 ,  0.02899971,  0.13634239, -0.00897989, -0.01212446],
       [ 0.23700127,  0.04412685, -0.6887047 , -0.01588956, -0.01352016]],
      dtype=float32)

In [4]:
# Global feature ranking: mean |importance| per feature (a magnitude view
# for ranking only, like SHAP summary plots — the explanations themselves
# are signed). Features 0 and 2 should dominate.
mean_imp = np.abs(importances).mean(axis=0)
for i, v in enumerate(mean_imp):
    print(f"feature {i}: {v:.4f}")

feature 0: 1.1176
feature 1: 0.0231
feature 2: 0.7197
feature 3: 0.0258
feature 4: 0.0324


## Fidelity vs. number of revealed features

`score_ordering` reveals features most-important-first and measures how well
the partially-revealed RoT reproduces the black box's labels.

In [5]:
import torch

x_t = torch.from_numpy(X)
y_t = torch.from_numpy(y.astype(np.int64))
order = rot._explainer_model.get_order(x_t)
acc = rot._explainer_model.score_ordering(x_t, y_t, order)
print("accuracy after revealing k=1..d features:")
print(acc.numpy())

accuracy after revealing k=1..d features:
[0.5035 0.982  0.9835 0.977  0.9785 0.9785]
